In [ ]:
# Importing Dependencies

In [2]:
import pandas as pd
import os
import glob

In [3]:
dataset = [pd.read_csv(f) for f in glob.glob(r"C:\Users\user\Downloads\CodeConnect_DataAnalyticsModule_participant_data\*.csv")]

In [5]:
dataset

[      customer_id       full_name                        email         phone  \
 0         C000001      Nida Memon     nida.memon6133@yahoo.com  0343-4328042   
 1         C000002   rizwan ansari  rizwan.ansari7047@gmail.com   03292755530   
 2         C000003       Noor Baig       noor.baig763@gmail.com   330 2812162   
 3         C000004   Naveed Hashmi  naveed.hashmi9719@gmail.com   03128373694   
 4         C000005   Mahnoor Mirza  mahnoor.mirza5501@yahoo.com   03251346045   
 ...           ...             ...                          ...           ...   
 44995     C044996   Kashif Abbasi  KASHIF.ABBASI9556@GMAIL.COM   03424368175   
 44996     C044997      Sana Javed   SANA.JAVED1711@HOTMAIL.COM  0334-6227664   
 44997     C044998     Anum Abbasi    anum_abbasi6678@gmail.com   03184589695   
 44998     C044999      Rabia Raza     rabia.raza3918@gmail.com           NaN   
 44999     C045000       iqra baig      IQRA.BAIG3854@GMAIL.COM  0313 5225659   
 
       gender registration

In [ ]:
#STEP: 1- AUDIT ALL 9 FILES to record row counts, column types, nulls, exact duplicate rows, and duplicate IDs for every file

In [7]:
for i, df in enumerate(dataset):
    print(f"\n{'='*40}\nFILE {i}\n{'='*40}")
    print("Rows:", len(df))
    print("\nColumn types:\n", df.dtypes)
    print("\nNulls per column:\n", df.isnull().sum())
    print("\nExact duplicate rows:", df.duplicated().sum())

    id_cols = [c for c in df.columns if c.lower() == 'id'
               or c.lower().endswith('_id')]

    for col in id_cols:
        print(f"Duplicate {col} values:", df[col].duplicated().sum())


FILE 0
Rows: 45000

Column types:
 customer_id           object
full_name             object
email                 object
phone                 object
gender                object
registration_date     object
city_id              float64
birth_year           float64
dtype: object

Nulls per column:
 customer_id             0
full_name               0
email                1698
phone                3599
gender               2199
registration_date       0
city_id              2233
birth_year           4600
dtype: int64

Exact duplicate rows: 0
Duplicate customer_id values: 0
Duplicate city_id values: 44349

FILE 1
Rows: 650

Column types:
 city_id              int64
city                object
province            object
region              object
latitude           float64
longitude          float64
population_tier     object
dtype: object

Nulls per column:
 city_id             0
city                0
province           25
region              0
latitude           33
longitude          31

In [ ]:
#STAGE: 2 - DATE VALIDATION to check unparseable dates, missing dates, dates outside the required range, and displays the invalid records.

In [9]:
for i, df in enumerate(dataset):
    date_cols = [c for c in df.columns if 'date' in c.lower()]
    print(f"\n{'='*50}\nFILE {i}\n{'='*50}")

    for col in date_cols:
        raw = df[col]
        dates = pd.to_datetime(
            raw, format='mixed', dayfirst=True, errors='coerce'
        )

        unparseable = raw.notna() & dates.isna()
        out_of_range = dates.notna() & (
            (dates < '2024-01-01') | (dates >= '2025-07-01')
        )

        print(f"\nColumn: {col}")
        print("Missing:", raw.isna().sum())
        print("Unparseable:", unparseable.sum())
        print("Out of range:", out_of_range.sum())

        if unparseable.any():
            print("Unparseable values:")
            print(raw[unparseable].unique()[:10])

        if out_of_range.any():
            print("Out-of-range values:")
            print(raw[out_of_range].unique()[:10])


FILE 0

Column: registration_date
Missing: 0
Unparseable: 252
Out of range: 8788
Unparseable values:
['2024-02-30' '2024-06-31' '2023-02-30' '2024-04-31' '2025-06-31'
 '2023-06-31' '2025-02-30' '00/00/2024' '00/00/2023' '00/00/2025']
Out-of-range values:
['2023-11-06' '2023-04-11' '2023-12-26' '2023-12-29' '2023-11-30'
 '2023-07-21' '2023-12-12' '2023-04-10' '2023-12-22' '13/11/2023']

FILE 1

FILE 2

Column: order_date
Missing: 0
Unparseable: 911
Out of range: 491
Unparseable values:
['00/00/2024' '2024-02-30' '2025-04-31' '2024-04-31' '2025-06-31'
 '2024-06-31' '00/00/2025' '2025-02-30' '2026-04-31' '2027-02-30']
Out-of-range values:
['2026-08-21' '2026-05-14' '2027-03-27' '2027-01-10' '2027-04-17'
 '2027-02-06' '09/10/2026' 'Nov 9 2026' '2027-06-06' '2026-06-05']

FILE 3

FILE 4

Column: payment_date
Missing: 5017
Unparseable: 871
Out of range: 793
Unparseable values:
['2024-02-30' '00/00/2025' '2025-06-31' '00/00/2024' '2025-02-30'
 '2024-06-31' '2024-04-31' '2025-04-31']
Out-of-r

In [ ]:
#STAGE-3: DISTINCE VALUES OF CATEGORIAL COLUMNS to spot spelling variants.

In [11]:
for i, df in enumerate(dataset):
    for col in categorical_cols:
        if col in df.columns:
            values = df[col].dropna().astype(str).unique()
            normalized = pd.Series(values).str.strip().str.lower()

            if normalized.duplicated().any() or any(v != v.strip() for v in values):
                print(f"\nFile {i} | Column: {col}")
                print(values)


File 1 | Column: city
['PurPurPur' 'SaraiTandoKot' 'WalaSaraiBasti' 'ShahChakAbad' 'RahKila'
 'AliDera' 'KalanMian' 'SaraiJand' 'BastiPur' 'BastiJand' 'KHURDSHAHCHAK'
 'ThalShah' 'GulGarhNagar' 'BastiKhurdMari' 'GulDeraKhan' 'TandoKalan'
 'BhalMariAli' 'DhokNau' 'GarhMor' 'AbadGulNagar' 'RahBasti' 'AbadRah'
 'RahMianMian' 'KhurdPind' 'MianKalanGul' 'MianJandTando' 'KhurdBhal'
 'KilaWalaNagar' 'PindWalaJand' 'Sahiwal' 'Bahawalpur' 'RahLalKhurd'
 'JandDera' 'PurAli' 'PurMariKalan' 'MorMianGola' 'PurDeraPur' 'KhanPind'
 'saraikhanrah' 'AliMorBhal' 'TandoBhalLal' 'BastiNauKalan' 'GulBastiWala'
 'MorPind' 'TandoMorLal' 'MianPur' 'KhurdLalGul' 'AliBhal' 'MariGulKila'
 'ThalMianGola' 'JandRahBasti' 'KhanKilaRah' 'WalaMari' 'BastiJandKhurd'
 'GulBastiLal' 'Kohat' 'TandoLal' 'ThalNauGola' 'ThalTando' 'GolaKotTando'
 'PurTandoDhok' 'BhalDeraKhurd' 'KalanBhalAbad' 'NagarDhokSarai'
 'PINDKOTNAGAR' 'ThalWala' 'ThalBhalGul' 'TandoNau' 'DeraWalaGola'
 'PindDhok' 'ShahTando' 'ALIGULBASTI' 'KilaMor' '

In [ ]:
#CHECK OUTLIERS ACROSS ALL 9 FILES -> thesee are the files and columns that we need to check for the outliers

In [15]:
for i, df in enumerate(dataset):
    print(f"\nFile {i}: {df.columns.tolist()}")


File 0: ['customer_id', 'full_name', 'email', 'phone', 'gender', 'registration_date', 'city_id', 'birth_year']

File 1: ['city_id', 'city', 'province', 'region', 'latitude', 'longitude', 'population_tier']

File 2: ['order_id', 'customer_id', 'order_date', 'order_status', 'ship_city_id', 'order_total', 'discount_code', 'channel']

File 3: ['order_item_id', 'order_id', 'product_id', 'seller_id', 'quantity', 'unit_price', 'discount_amount']

File 4: ['payment_id', 'order_id', 'payment_method', 'payment_status', 'amount', 'payment_date']

File 5: ['product_id', 'product_name', 'category', 'sub_category', 'brand', 'list_price', 'cost_price', 'weight_kg', 'seller_id']

File 6: ['return_id', 'order_item_id', 'return_date', 'return_reason', 'return_status', 'refund_amount']

File 7: ['seller_id', 'seller_name', 'join_date', 'seller_tier', 'city_id', 'rating']

File 8: ['shipment_id', 'order_id', 'courier', 'shipped_date', 'delivered_date', 'shipping_fee', 'delivery_status']


In [ ]:
#we have to identify ourselves as to which fields can have outliers

In [ ]:
#possibility:  File 0: birth_year, File 1: latitude, longitude, File 2: order_total,  #File 3: quantity, unit_price,discount_amount
               #File 4: amount, File 5: list_price, cost_price, weight_kg, File 6: refund_amount, File 7: rating
               # File 8: Shipping Fee


In [ ]:
#ORPHAN KEYS

In [16]:
relationships = [
    (2, 'customer_id', 0, 'customer_id'),       # Orders → Customers
    (2, 'ship_city_id', 1, 'city_id'),          # Orders → Cities
    (3, 'order_id', 2, 'order_id'),             # Order items → Orders
    (3, 'product_id', 5, 'product_id'),         # Order items → Products
    (3, 'seller_id', 7, 'seller_id'),            # Order items → Sellers
    (4, 'order_id', 2, 'order_id'),             # Payments → Orders
    (5, 'seller_id', 7, 'seller_id'),            # Products → Sellers
    (6, 'order_item_id', 3, 'order_item_id'),   # Returns → Order items
    (7, 'city_id', 1, 'city_id'),               # Sellers → Cities
    (8, 'order_id', 2, 'order_id')              # Shipments → Orders
]

for child_idx, child_col, parent_idx, parent_col in relationships:
    child = dataset[child_idx]
    parent = dataset[parent_idx]

    orphan_mask = (
        child[child_col].notna() &
        ~child[child_col].isin(parent[parent_col])
    )

    print(
        f"File {child_idx} ({child_col}) → "
        f"File {parent_idx} ({parent_col}): "
        f"{orphan_mask.sum()} orphan rows"
    )

    if orphan_mask.any():
        print(child.loc[orphan_mask, [child_col]].head())

File 2 (customer_id) → File 0 (customer_id): 330 orphan rows
    customer_id
197      C98859
455      C96770
677      C96644
814      C96819
913      C91906
File 2 (ship_city_id) → File 1 (city_id): 0 orphan rows
File 3 (order_id) → File 2 (order_id): 0 orphan rows
File 3 (product_id) → File 5 (product_id): 932 orphan rows
    product_id
127     P96243
167     P97524
234     P93742
554     P90073
561     P99790
File 3 (seller_id) → File 7 (seller_id): 0 orphan rows
File 4 (order_id) → File 2 (order_id): 0 orphan rows
File 5 (seller_id) → File 7 (seller_id): 0 orphan rows
File 6 (order_item_id) → File 3 (order_item_id): 139 orphan rows
    order_item_id
54        OI94118
88        OI92084
316       OI99844
471       OI90501
898       OI95135
File 7 (city_id) → File 1 (city_id): 0 orphan rows
File 8 (order_id) → File 2 (order_id): 0 orphan rows


In [ ]:
#To check records with missing relationship ids, we check missing foreign keys

In [17]:
#This identifies records with missing relationship IDs separately from orphan keys

for child_idx, child_col, parent_idx, parent_col in relationships:
    df = dataset[child_idx]
    missing = df[child_col].isna().sum()

    print(f"File {child_idx} | {child_col} | Missing IDs: {missing}")

File 2 | customer_id | Missing IDs: 0
File 2 | ship_city_id | Missing IDs: 6516
File 3 | order_id | Missing IDs: 0
File 3 | product_id | Missing IDs: 0
File 3 | seller_id | Missing IDs: 0
File 4 | order_id | Missing IDs: 0
File 5 | seller_id | Missing IDs: 0
File 6 | order_item_id | Missing IDs: 0
File 7 | city_id | Missing IDs: 0
File 8 | order_id | Missing IDs: 0


In [ ]:
#Possible duplicate customers and products

In [19]:
#This finds records matching on all four fields. Treat them as possible duplicates to investigate, not automatically confirmed duplicates.
#CUSTOMERS: 
customers = dataset[0].copy()

customers['normalized_name'] = (
    customers['full_name'].str.strip().str.lower()
)
customers['normalized_email'] = customers['email'].str.strip().str.lower()
customers['normalized_phone'] = (
    customers['phone'].astype('string').str.replace(r'\D', '', regex=True)
)

duplicate_customers = customers[
    customers.duplicated(
        subset=['normalized_email', 'normalized_phone',
                'birth_year', 'normalized_name'],
        keep=False
    )
]

duplicate_customers.sort_values('normalized_email')

,customer_id,full_name,email,phone,gender,registration_date,city_id,birth_year,normalized_name,normalized_email,normalized_phone
43555,C043556,Ahmed Akhtar,AHMED.AKHTAR4584@GMAIL.COM,03437275239,NaN,2024-01-29,407.0,1993.0,ahmed akhtar,ahmed.akhtar4584@gmail.com,03437275239
40740,C040741,Ahmed Akhtar,ahmed.akhtar4584@gmail.com,0343-7275239,M,2024-01-11,407.0,1993.0,ahmed akhtar,ahmed.akhtar4584@gmail.com,03437275239
42680,C042681,Ahmed Ansari,ahmed.ansari8680@gmail.com,0320-8896124,male,Jun 5 2025,450.0,1990.0,ahmed ansari,ahmed.ansari8680@gmail.com,03208896124
28913,C028914,Ahmed Ansari,ahmed.ansari8680@gmail.com,0320-8896124,M,2024-10-01,450.0,1990.0,ahmed ansari,ahmed.ansari8680@gmail.com,03208896124
42668,C042669,Ahmed Butt,ahmed.butt6659@gmail.com,03436439180,NaN,2025-02-20,501.0,1995.0,ahmed butt,ahmed.butt6659@gmail.com,03436439180
...,...,...,...,...,...,...,...,...,...,...,...
41181,C041182,Laiba Chaudhry,NaN,0335 1601291,F,2025-06-08,617.0,1998.0,laiba chaudhry,NaN,03351601291
42517,C042518,Laiba Chaudhry,NaN,0335-1601291,F,2024-02-08,617.0,1998.0,laiba chaudhry,NaN,03351601291
43106,C043107,Hussain Siddiqui,NaN,0342-4907675,M,2024-04-19,614.0,1989.0,hussain siddiqui,NaN,03424907675
43878,C043879,Ayesha Nawaz,NaN,03165770670,f,20/03/2025,450.0,1982.0,ayesha nawaz,NaN,03165770670


In [20]:
#PRODUCTS

products = dataset[5].copy()

products['normalized_name'] = products['product_name'].str.strip().str.lower()
products['normalized_brand'] = products['brand'].str.strip().str.lower()

duplicate_products = products[
    products.duplicated(
        subset=['normalized_name', 'normalized_brand'],
        keep=False
    )
]

duplicate_products.sort_values(['normalized_name', 'normalized_brand'])

,product_id,product_name,category,sub_category,brand,list_price,cost_price,weight_kg,seller_id,normalized_name,normalized_brand
1406,P01407,BeauPro Trimmer Y812,Beauty & Personal Care,Beauty Premium,BeauPro,400,188.0,0.62,S0057,beaupro trimmer y812,beaupro
3164,P03165,BeauPro Trimmer Y812,BEAUTY & PERSONAL CARE,Beauty Premium,BeauPro,393,188.0,0.62,S0193,beaupro trimmer y812,beaupro
62,P00063,ElecPro Power Strip D878,electronics,NaN,ElecPro,6400,5574.0,2.21,S0329,elecpro power strip d878,elecpro
3084,P03085,ElecPro Power Strip D878,ELECTRONICS,Electronics Essentials,ElecPro,6323,9484.0,2.21,S0134,elecpro power strip d878,elecpro
2215,P02216,ElecPro Power Strip U248,Electronic Items,Electronics Essentials,ElecPro,55470,83205.0,NaN,S0378,elecpro power strip u248,elecpro
3089,P03090,ElecPro Power Strip U248,Elec.,Electronics Essentials,ElecPro,54644,81966.0,1.27,S0251,elecpro power strip u248,elecpro
871,P00872,Generic Laptop Stand P215,Electronics,Electronics Premium,Generic,10580,8314.0,1.64,S0329,generic laptop stand p215,generic
3149,P03150,Generic Laptop Stand P215,electronics,Electronics Premium,Generic,10805,8314.0,1.64,S0349,generic laptop stand p215,generic
1366,P01367,Generic Polo Shirt X880,Fashion,Fashion Premium,Generic,5220,2696.0,0.54,S0286,generic polo shirt x880,generic
3124,P03125,Generic Polo Shirt X880,Clothing,Fashion Premium,Generic,5254,2696.0,0.54,S0286,generic polo shirt x880,generic


In [ ]:
#Consistency checks

In [33]:
#CHANGED DATATYPE INTO NUMERIC
items['quantity'] = pd.to_numeric(items['quantity'], errors='coerce')

items['unit_price'] = pd.to_numeric(
    items['unit_price'].astype(str).str.replace(r'Rs\.?\s*|,', '', regex=True),
    errors='coerce'
)

items['discount_amount'] = pd.to_numeric(
    items['discount_amount'], errors='coerce'
)

In [42]:
items['line_total'] = (
    items['quantity'] * items['unit_price']
    - items['discount_amount']
)
calculated_totals = items.groupby('order_id')['line_total'].sum()
orders['order_total'] = pd.to_numeric(
    orders['order_total'].astype(str).str.replace(r'Rs\.?\s*|,', '', regex=True),
    errors='coerce'
)
orders['difference'] = orders['order_total'] - orders['calculated_total']
orders[orders['difference'].abs() > 0.01][
    ['order_id', 'order_total', 'calculated_total', 'difference']
]



,order_id,order_total,calculated_total,difference
2,O0083748,2362,2023.0,339.0
16,O0042918,1476,561.0,915.0
21,O0025356,15682,0.0,15682.0
24,O0051382,4590,3623.0,967.0
41,O0130009,3080,1156.0,1924.0
...,...,...,...,...
163160,O0091604,3406,1060.0,2346.0
163173,O0121296,6882,0.0,6882.0
163188,O0096974,5197,3846.0,1351.0
163195,O0014965,3271,0.0,3271.0


In [ ]:
#The table shows that 33,686 orders have totals that don't match the totals calculated from their order items.

In [40]:
(orders['difference'].abs() > 0.01).sum()

np.int64(33686)

In [ ]:
#B. Does discount_code map to a percentage?

In [24]:

#Which discount codes exist and how often they appear
dataset[2]['discount_code'].value_counts(dropna=False)

discount_code
NaN        112059
SAVE15       7846
SAVE10       7352
SAVE20       6763
SAVE40       5898
SAVE30       5246
SAVE5        5066
SAVE25       4112
SAVE50       2675
Save15        369
 SAVE15       348
save20        343
 SAVE10       336
Save10        334
save15        330
save10        322
 SAVE20       316
save40        292
Save20        286
 SAVE40       260
 SAVE5        258
Save40        255
save5         236
Save30        236
save30        235
 SAVE30       231
Save5         230
Save25        201
save25        192
 SAVE25       188
 SAVE50       162
Save50        122
save50        101
Name: count, dtype: int64

In [ ]:
#It tells you how many times each discount code appears in the orders dataset, including missing values (NaN).

In [47]:
items = dataset[3]

print(items[['quantity', 'unit_price', 'discount_amount']].head(10))

  quantity unit_price  discount_amount
0        1  Rs. 2,290              0.0
1        1       5850              0.0
2        1       1900              0.0
3        1       2180              0.0
4        1       2100              0.0
5        1       6340              0.0
6        2       1840              0.0
7        2       3570              NaN
8        1       1340            268.0
9        1        650              0.0


In [48]:
#Prepared items data by converting quantity, unit_price and discount into numeric values and Calculated gross amount and discount percentage
items = dataset[3].copy()

items['quantity'] = pd.to_numeric(items['quantity'], errors='coerce')
items['unit_price'] = pd.to_numeric(
    items['unit_price'].astype(str).str.replace(r'Rs\.?\s*|,', '', regex=True),
    errors='coerce'
)

items['discount_amount'] = pd.to_numeric(
    items['discount_amount'], errors='coerce'
)

items['gross_amount'] = items['quantity'] * items['unit_price']

items['discount_percentage'] = (
    items['discount_amount'] / items['gross_amount'] * 100
)

items[['quantity', 'unit_price', 'discount_amount',
       'gross_amount', 'discount_percentage']].head(10)

,quantity,unit_price,discount_amount,gross_amount,discount_percentage
0,1.0,2290.0,0.0,2290.0,0.0
1,1.0,5850.0,0.0,5850.0,0.0
2,1.0,1900.0,0.0,1900.0,0.0
3,1.0,2180.0,0.0,2180.0,0.0
4,1.0,2100.0,0.0,2100.0,0.0
5,1.0,6340.0,0.0,6340.0,0.0
6,2.0,1840.0,0.0,3680.0,0.0
7,2.0,3570.0,NaN,7140.0,NaN
8,1.0,1340.0,268.0,1340.0,20.0
9,1.0,650.0,0.0,650.0,0.0


In [49]:
items['discount_percentage'].round(2).value_counts().sort_index()

discount_percentage
-130.00       1
-129.98       1
-50.02        1
-50.01        1
-50.00       12
             ..
 14000.53     1
 14001.86     1
 14002.09     1
 36373.52     1
 inf         83
Name: count, Length: 532, dtype: int64

In [50]:
#Checked invalid values:

print("Zero or missing gross amounts:", (items['gross_amount'].isna() | (items['gross_amount'] == 0)).sum())
print("Negative discounts:", (items['discount_amount'] < 0).sum())
print("Discounts greater than gross amount:", (items['discount_amount'] > items['gross_amount']).sum())

Zero or missing gross amounts: 1295
Negative discounts: 0
Discounts greater than gross amount: 1586


In [51]:
#Filtered invalid amounts
valid = items[
    (items['gross_amount'] > 0) &
    (items['discount_amount'] >= 0) &
    (items['discount_amount'] <= items['gross_amount'])
].copy()

valid['discount_percentage'] = (
    valid['discount_amount'] / valid['gross_amount'] * 100
).round().astype(int)

valid['discount_percentage'].value_counts().sort_index()

discount_percentage
0     190214
1          3
5       9858
10     14051
15     14882
20     13049
25      7900
28         1
30     10052
40     14319
50      6578
Name: count, dtype: int64

In [55]:
#Compared percentage patterns with codes

check = dataset[2][['order_id', 'discount_code']].merge(
    valid[['order_id', 'discount_percentage']],
    on='order_id',
    how='inner'
)

check.groupby(['discount_code', 'discount_percentage']).size().unstack(fill_value=0)


discount_percentage,0%%,1%%,10%%,15%%,20%%,25%%,28%%,30%%,40%%,5%%,50%%
discount_code,,,,,,,,,,,
SAVE10,0,0,551,0,0,0,0,0,0,0,0
SAVE15,2,0,0,566,0,0,0,0,0,0,0
SAVE20,2,0,0,0,546,0,0,0,0,0,0
SAVE25,1,0,0,0,0,330,0,0,0,0,0
SAVE30,1,0,0,0,0,0,0,400,0,0,0
SAVE40,1,0,0,0,0,0,0,0,545,0,0
SAVE5,2,0,0,0,0,0,0,0,0,434,0
SAVE50,1,0,0,0,0,0,0,0,0,0,358
SAVE10,56,0,12666,0,0,0,0,0,0,0,0
